# Loop Exercise

When profiling the original code, we see the innermmost loop accounts for almost all of the time the code spends running, so we should begin there. In this innermost loop, we add `math.tan(my_list[i]) + temp_var` in each iteration of the loop, and this value doesn't change as `k` changes. So we can, instead, add this value multiplied by 100 in the loop over `j` instead and this will give the same value.

In the innermost loop over `k` we also added the value `k`. This means we're adding $\sum\limits_{k=0}^{99}$, which we can evaluate using the arithmetic sum formula to be $\frac{100(0+99)}{2} = 4950$. This means we can add this constant value once in the `j` loop and get the same answer.

This allows us to entirely eliminate this innermost loop. We see this immediately reduces the time taken for the function to run by a factor of ~100.

In [ ]:
%load_ext line_profiler
import math

def loopy_function():

    my_list = []

    for i in range(100):
        my_list.append(i ** 2)

    result = 0

    for i in range(100):
        for j in range(100):
            temp_var = math.sqrt(j)
            result = result + 100 * (math.tan(my_list[i]) + temp_var) + 4950

    return result

%lprun -f loopy_function print(loopy_function())

Next, we can work to simplify and remove the inner loop. In this loop we are currently adding `100 * math.tan[i] + 4950` which is not a function of `j`, so we can multiply this value by 100 and add it in the outermost loop over `i` instead.

The term involving `temp_var` is a little trickier. As it is the sum of the sum function applied to each value between 0 and 99 inclusive, we can replace is with a sum of a map applying `math.sqrt` to a range in the outermost loop over `i`.

In [ ]:
%load_ext line_profiler
import math

def loopy_function():

  my_list=[]

  for i in range(100):
    my_list.append(i**2)

  result = 0

  for i in range(100):
    result = result + 10000 * (math.tan(my_list[i])) + 495000
    result = result + 100 * sum(map(math.sqrt, range(100)))

  return result

%lprun -f loopy_function print(loopy_function())

Finally, we can work on the final remaining loop. The term that applies `math.tan` to each entry in `my_list` can be replaced with a sum of a map. We can also replace the definition of `my_list` with a list comprehension, further speeding up the creation of this list.

The term `495000` was previously added 100 times, we can simply add `49500000` instead.

The term `100 * sum(map(math.sqrt, range(100)))` isn't a function of `i`, so we can simply multiply this by 100 and add it outside of the loop.

In [ ]:
#@title
# The third optimisation is to combine the two remaining loops and replace them with a map function that we take the sum of
# We use a list comprehension to form the list passed to the map
# The resultant function is approximately 10,000 times faster than the function we began with
%load_ext line_profiler
import math

def loopy_function():
    result = 10000 * sum(map(math.tan, [i ** 2 for i in range(100)])) + 49500000
    result = result + 10000 * sum(map(math.sqrt, range(100)))

    return result

%lprun -f loopy_function print(loopy_function())

The culmination of all of these optimisations is to make the code about 10,000 times faster than the original.